# RK1828 Chinese-CLIP 图文匹配 Demo


[Chinese-CLIP](https://github.com/OFA-Sys/Chinese-CLIP)（达摩院）中文图文匹配：
给定一张图和几条中文候选描述，计算匹配概率。文本编码器和视觉编码器分别转换、
全部跑在 RK1828 上。示例图上正确描述的得分约 0.96 vs 0.04。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 运行图文匹配 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("chinese-clip-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/chinese_clip/demo_x86_64.tar.gz"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.embed.bin", "demos/chinese_clip/model/chinese_clip_text.embed.bin"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.rknn", "demos/chinese_clip/model/chinese_clip_text.rknn"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.tokenizer.gguf", "demos/chinese_clip/model/chinese_clip_text.tokenizer.gguf"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_text.weight", "demos/chinese_clip/model/chinese_clip_text.weight"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_vision.rknn", "demos/chinese_clip/model/chinese_clip_vision.rknn"),
    ("chinese-clip-rknn3-rk1828", "model/chinese_clip_vision.weight", "demos/chinese_clip/model/chinese_clip_vision.weight"),
    ("chinese-clip-rknn3-rk1828", "model/demo.jpg", "demos/chinese_clip/model/demo.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/chinese_clip/demo_x86_64.tar.gz", "demos/chinese_clip")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 运行图文匹配

输入示例图 `model/demo.jpg`，demo 内置几条中文候选描述，输出每条的概率。

In [ ]:
show_images(["demos/chinese_clip/model/demo.jpg"], max_width=400)

In [ ]:
DEMO = "demos/chinese_clip"
run_demo(["./rknn_qaclipvit_demo",
          "model/chinese_clip_text.rknn", "model/chinese_clip_text.weight",
          "model/chinese_clip_text.embed.bin", "model/chinese_clip_text.tokenizer.gguf",
          "model/chinese_clip_vision.rknn", "model/chinese_clip_vision.weight",
          "model/demo.jpg", "1"], DEMO)

## 小结

- CLIP 类模型把图文映射到同一向量空间，"匹配"就是算相似度，可做零样本分类/以文搜图。
- 多模态模型的文本侧（tokenizer + embed）也需要一起部署，见 `model/chinese_clip_text.*`。